In [1]:
import numpy as np
import pandas as pd
import json
import time
import datetime
import calendar

In [2]:
# load in psiturk data
rm1df = pd.read_json('../../data/db/exported/room1-2.8.19.json')
rm2df = pd.read_json('../../data/db/exported/room2-2.8.19.json')

# drop runs that didn't finish
rm1df = rm1df[rm1df.status != 1]
rm2df = rm2df[rm2df.status != 1]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)
rm2df = rm2df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)

# convert datastring to dict 
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# add test room column
rm1df['testroom'] = 1
rm2df['testroom'] = 2

# concatenate dataframes
expdf = pd.concat([rm1df, rm2df], ignore_index=True)

In [3]:
expdf

,uniqueid,datastring,beginhit,endhit,hitid,status,testroom
0,debugrcDp9:debugAniFb,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-02 22:31:54.578217,2018-10-02 23:35:38.071920,debugvPYXO,3,1
1,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 18:14:31.394716,2018-10-12 19:12:06.999050,debug7rmxU,3,1
2,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 19:17:03.503575,2018-10-12 20:10:44.883411,debugTkKFp,3,1
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 20:22:07.319435,2018-10-12 21:09:46.566325,debugonOYk,3,1
4,debugGaDml:debugFTHoY,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 21:28:01.305584,2018-10-12 22:16:31.057617,debugXHY6O,3,1
5,debugGVTD3:debugfpzCT,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 22:59:26.829625,2018-10-13 00:01:24.711451,debuggQ0y6,3,1
6,debugokLIG:debugalG88,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:28:34.378931,2018-10-13 19:15:12.977411,debugslG65,3,1
7,debugdhnfF:debug6fW93,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 19:35:43.889748,2018-10-13 20:32:37.663906,debugszpCm,3,1
8,debugHKLdw:debugk43rK,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 20:37:43.034617,2018-10-13 21:27:38.367721,debugHGwAN,3,1
9,debugnj3ww:debugAC3on,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 21:34:31.560211,2018-10-13 22:25:03.538624,debughqOmP,3,1


In [10]:
# load pre/post questionnaire responses
preqdf = pd.read_csv('../../data/google-form-data/Pre-experiment Questionnaire.csv', parse_dates=[0])
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
postqdf = pd.read_csv('../../data/google-form-data/Post-experiment questionnaire.csv', parse_dates=[0])
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})

# exclude test runs
preqdf = preqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)
postqdf = postqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)

# convert form timestamp to POSIX time **SERIES.APPLY(DT.DT.TIMESTAMP).MULTIPLY(1000) DOES NOT WORK**
newpretimestamp = pd.Series([0]*len(preqdf['preqtime']))
for ix, val in enumerate(newpretimestamp):
    newpretimestamp[ix] = preqdf['preqtime'][ix].timestamp()*1000
preqdf['preqtime'] = newpretimestamp

newposttimestamp = pd.Series([0]*len(postqdf['postqtime']))
for ix, val in enumerate(newposttimestamp):
    newposttimestamp[ix] = postqdf['postqtime'][ix].timestamp()*1000
postqdf['postqtime'] = newposttimestamp

### for mapping between Google Forms with experiment IDs and SQLite databases with PsiTurk IDs

In [30]:
def concat_data(turkdata, preform, postform):
    """
    matches between experiment data and pre/post questionnaire responses, concatenates dataframes
    """
    turktimes = {}
    preqtimes = {}
    for ix, sub in turkdata.iterrows():
        turktimes[ix] = sub['datastring']['data'][0]['dateTime']
    for ix, sub in preform.iterrows():
        preqtimes[ix] = preform['preqtime'][sub]
        
    for ix, subtime in turktimes.items():
        
        

SyntaxError: unexpected EOF while parsing (<ipython-input-30-d88114952d53>, line 10)

In [36]:
# add empty columns to 
expdf = pd.concat([expdf, pd.DataFrame(columns=newcols)], sort=False)

turktimes = {}
preqtimes = {}

# get psiturk start times
for ix, sub in expdf.iterrows():
    turktimes[sub['datastring']['data'][0]['dateTime']] = ix
# get gform submit times
for ix, sub in preqdf.iterrows():
    preqtimes[preqdf['preqtime'][sub]] = ix

# iterate over psiturk start times, find closest gform time
for turktime, ix in turktimes.items():
    closest = preqtimes.get(turktime, preqtimes[min(preqtimes.keys(), key=lambda k: abs(k-turktime))])
    



ValueError: cannot index with vector containing NA / NaN values

In [42]:
type(preqdf.loc[0])

pandas.core.series.Series

In [53]:
expdf.reindex([expdf, pd.DataFrame(columns=newcols)])

,uniqueid,datastring,beginhit,endhit,hitid,status,testroom,preqtime,Subject ID,"Outside of this study, have you ever watched an episode of either of the TV shows ""Atlanta"" or ""Arrested Development?""",...,What is/was your major?,How many hours of sleep did you get last night?,How many cups of coffee have you had today?,How alert are you feeling?,postqtime,How engaging did you find the episode?,How easy/difficult was it to follow the episode?,How well do you feel you recalled the events of the episode?,How well do you feel you learned the characters' names over the course of the episode?,How tired do you feel?
0,debugrcDp9:debugAniFb,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-02 22:31:54.578217,2018-10-02 23:35:38.071920,debugvPYXO,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 18:14:31.394716,2018-10-12 19:12:06.999050,debug7rmxU,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 19:17:03.503575,2018-10-12 20:10:44.883411,debugTkKFp,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 20:22:07.319435,2018-10-12 21:09:46.566325,debugonOYk,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,debugGaDml:debugFTHoY,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 21:28:01.305584,2018-10-12 22:16:31.057617,debugXHY6O,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,debugGVTD3:debugfpzCT,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 22:59:26.829625,2018-10-13 00:01:24.711451,debuggQ0y6,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,debugokLIG:debugalG88,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:28:34.378931,2018-10-13 19:15:12.977411,debugslG65,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,debugdhnfF:debug6fW93,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 19:35:43.889748,2018-10-13 20:32:37.663906,debugszpCm,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,debugHKLdw:debugk43rK,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 20:37:43.034617,2018-10-13 21:27:38.367721,debugHGwAN,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,debugnj3ww:debugAC3on,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 21:34:31.560211,2018-10-13 22:25:03.538624,debughqOmP,3.0,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
newcols = pd.unique(np.concatenate([i.columns.values for i in [preqdf,postqdf]]))

In [94]:
np.unique(preqdf.columns.values)

array(['Are you taking any medications or have you had any recent injuries that could affect your memory or attention? (if so, describe below)',
       'Do you have any hearing or speech impairments?',
       'Do you have normal color vision?', 'Ethnicity',
       'Highest Degree Achieved', 'How alert are you feeling?',
       'How many cups of coffee have you had today?',
       'How many hours of sleep did you get last night?',
       'If yes above, describe',
       'If you are currently an undergraduate, what year are you?',
       'In what year were you born?', 'Is English your first language?',
       'Outside of this study, have you ever watched an episode of either of the TV shows "Atlanta" or "Arrested Development?"',
       'Race (check all that apply)', 'Sex', 'Subject ID', 'Timestamp',
       'What is/was your major?'], dtype=object)